# Merge BERTopic Models Across Outlets

This notebook loads the saved BERTopic models for Tagesschau, RT, Antispiegel, Tichys Einblick, Nius, Compact, and Deutschlandkurier, merges them with `BERTopic.merge_models(...)`, then builds merged article-level UMAP maps.


In [ ]:
import os
import sys
from pathlib import Path

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MIN_SIMILARITY = 0.7


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing .git")


PROJECT_ROOT = find_project_root(Path.cwd())
MODULE_ROOT = PROJECT_ROOT / "1a_BERTopic"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

os.environ["NUMBA_CACHE_DIR"] = str(PROJECT_ROOT / ".numba_cache")

MODEL_DIR_CANDIDATES = [
    PROJECT_ROOT / "1a_BERTopic" / "local_outputs",
    PROJECT_ROOT / "1a_BERTopic" / "outputs",
    PROJECT_ROOT / "BERTopic" / "outputs",
]
MERGED_SAVE_DIR = PROJECT_ROOT / "1a_BERTopic" / "local_outputs" / "merged_all_outlets_model"

print(f"Project root: {PROJECT_ROOT}")
print("Model path candidates:")
for candidate in MODEL_DIR_CANDIDATES:
    print(f"  - {candidate}")


In [ ]:
import importlib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from bertopic import BERTopic

import merged_outlets_analysis as moa
moa = importlib.reload(moa)

ALT_MEDIA_OUTLET_KEYS = getattr(
    moa,
    "ALT_MEDIA_OUTLET_KEYS",
    ("rt", "antispiegel", "tichys", "nius", "compact", "deutschlandkurier"),
)
OUTLET_SPECS = moa.OUTLET_SPECS
build_merged_article_frame = moa.build_merged_article_frame
build_outlet_topic_coverage_summary = moa.build_outlet_topic_coverage_summary
combine_prepared_documents = moa.combine_prepared_documents
load_all_prepared_documents = moa.load_all_prepared_documents
plot_merged_topic_umap = moa.plot_merged_topic_umap
plot_outlet_highlight_umap = moa.plot_outlet_highlight_umap
resolve_model_paths = moa.resolve_model_paths

MODEL_PATHS = resolve_model_paths(MODEL_DIR_CANDIDATES)

loaded_models = {
    key: BERTopic.load(model_path, embedding_model=EMBEDDING_MODEL)
    for key, model_path in MODEL_PATHS.items()
}

for key, model in loaded_models.items():
    topic_info = model.get_topic_info()
    print(f"Loaded {key}: {MODEL_PATHS[key]} ({len(topic_info)} rows in topic info)")

tm_ts = loaded_models["tagesschau"]
tm_rt = loaded_models["rt"]
tm_as = loaded_models["antispiegel"]
tm_te = loaded_models["tichys"]
tm_ns = loaded_models["nius"]
tm_cm = loaded_models["compact"]
tm_dk = loaded_models["deutschlandkurier"]


In [ ]:
models_to_merge = [
    tm_ts,
    tm_rt,
    tm_as,
    tm_te,
    tm_ns,
    tm_cm,
    tm_dk,
]

merged_model = BERTopic.merge_models(
    models_to_merge,
    min_similarity=MIN_SIMILARITY,
    embedding_model=EMBEDDING_MODEL,
)

merged_topic_info = merged_model.get_topic_info()
display(merged_topic_info.head(30))
print("Merged topic count:", len(merged_topic_info))


In [ ]:
import shutil

SAVE_MERGED_MODEL = False

if SAVE_MERGED_MODEL:
    MERGED_SAVE_DIR.parent.mkdir(parents=True, exist_ok=True)
    if MERGED_SAVE_DIR.exists():
        shutil.rmtree(MERGED_SAVE_DIR)
    merged_model.save(
        MERGED_SAVE_DIR,
        serialization="safetensors",
        save_ctfidf=True,
        save_embedding_model=EMBEDDING_MODEL,
    )
    print(f"Saved merged model to: {MERGED_SAVE_DIR}")
else:
    print("Skipped save. Set SAVE_MERGED_MODEL = True to persist the merged model.")


In [ ]:
import pandas as pd

prepared_by_outlet = load_all_prepared_documents(PROJECT_ROOT)
prepared_summary = pd.DataFrame(
    [
        {
            "Outlet": OUTLET_SPECS[key].label,
            "Prepared_Documents": len(df),
        }
        for key, df in prepared_by_outlet.items()
    ]
).sort_values("Outlet").reset_index(drop=True)
display(prepared_summary)

combined_prepared = combine_prepared_documents(prepared_by_outlet)
print("Combined prepared documents:", len(combined_prepared))


In [ ]:
merged_articles, merged_topic_info_display, merged_umap_model = build_merged_article_frame(
    merged_model,
    combined_prepared,
)

display(merged_articles[["outlet_label", "document_id", "merged_topic", "merged_display_label"]].head())
display(merged_articles.groupby("outlet_label").size().rename("Article_Count").reset_index())


In [ ]:
fig, ax = plot_merged_topic_umap(
    merged_articles,
    merged_topic_info_display,
    top_n=20,
)
plt.show()


In [ ]:
coverage_summary = build_outlet_topic_coverage_summary(
    merged_articles,
    merged_topic_info_display,
)
display(coverage_summary)

# Plot semantic footprint maps for ALL outlets, including Tagesschau
ALL_OUTLET_KEYS = ("tagesschau",) + ALT_MEDIA_OUTLET_KEYS

for outlet_key in ALL_OUTLET_KEYS:
    spec = OUTLET_SPECS[outlet_key]
    print(f"Plotting outlet coverage: {spec.label}")
    fig, ax = plot_outlet_highlight_umap(
        merged_articles,
        outlet_key,
        alt_media_only=False,
        show_kde=True,
        min_label_articles=15,
        merged_topic_info=merged_topic_info_display,
    )
    plt.show()

In [ ]:
# H1 thesis functions for agenda distortion analysis


def compute_outlet_topic_measures(merged_articles, min_articles=10) -> pd.DataFrame:
    """Computes four complementary measures of agenda distortion per outlet.

    Designed to separate TYPE A (breadth restriction) from TYPE B
    (concentration distortion).

    Thesis relevance:
        Directly operationalizes H1 (Agenda Distortion).

    Args:
        merged_articles: Article-level merged BERTopic assignments.
        min_articles: Minimum number of articles required for a topic to count
            as covered by an outlet.

    Returns:
        pd.DataFrame: Outlet-level H1 measures.
    """
    import math
    import warnings

    def _binomial_tail_prob_ge_k(n, p, k, scipy_state):
        if k <= 0:
            return 1.0
        if n < k or p <= 0.0:
            return 0.0
        if p >= 1.0:
            return 1.0

        try:
            if scipy_state["binom"] is None:
                from scipy.stats import binom

                scipy_state["binom"] = binom
            return float(1.0 - scipy_state["binom"].cdf(k - 1, n, p))
        except Exception:
            if not scipy_state["warned"]:
                warnings.warn(
                    "scipy is unavailable; using a recursive binomial tail fallback for coverage_breadth_expected.",
                    RuntimeWarning,
                )
                scipy_state["warned"] = True

            q = 1.0 - p
            log_p0 = n * math.log(q) if q > 0 else float("-inf")
            if log_p0 < -745:
                return 1.0

            pmf = math.exp(log_p0)
            cdf = pmf
            upper = min(k - 1, n)
            for j in range(0, upper):
                ratio = ((n - j) / (j + 1)) * (p / q) if q > 0 else float("inf")
                pmf *= ratio
                cdf += pmf
            return float(min(max(1.0 - cdf, 0.0), 1.0))

    topic_df = merged_articles.loc[
        merged_articles["merged_topic"] != -1,
        ["outlet_label", "merged_topic"],
    ].copy()
    columns = [
        "outlet_label",
        "n_articles",
        "corpus_share",
        "entropy",
        "entropy_normalized",
        "coverage_breadth_raw",
        "coverage_breadth_expected",
        "coverage_breadth_relative",
        "kl_from_tagesschau",
        "topic_dominance_score",
    ]
    if topic_df.empty:
        return pd.DataFrame(columns=columns)
        # H1 evidence type: BOTH

    all_outlets = sorted(merged_articles["outlet_label"].dropna().unique().tolist())
    all_topics = sorted(topic_df["merged_topic"].unique().tolist())
    total_topics = len(all_topics)

    outlet_topic_counts = (
        topic_df.groupby(["outlet_label", "merged_topic"])
        .size()
        .unstack(fill_value=0)
        .reindex(index=all_outlets, columns=all_topics, fill_value=0)
        .astype(float)
    )
    outlet_topic_totals = outlet_topic_counts.sum(axis=1).astype(int)
    outlet_article_totals = (
        merged_articles.groupby("outlet_label")
        .size()
        .reindex(all_outlets, fill_value=0)
        .astype(int)
    )
    topic_totals = outlet_topic_counts.sum(axis=0).astype(float)
    corpus_topic_total = float(outlet_topic_totals.sum())
    corpus_topic_share = topic_totals / corpus_topic_total if corpus_topic_total else topic_totals * 0.0

    tagesschau_label = "Tagesschau"
    if tagesschau_label not in outlet_topic_counts.index:
        raise KeyError("Tagesschau not found in merged_articles['outlet_label'].")

    tag_counts = outlet_topic_counts.loc[tagesschau_label].astype(float)
    tag_probs = (tag_counts + 1.0) / (float(tag_counts.sum()) + total_topics)

    scipy_state = {"binom": None, "warned": False}
    rows = []
    corpus_article_total = int(len(merged_articles))

    for outlet_label in all_outlets:
        counts = outlet_topic_counts.loc[outlet_label].astype(float)
        n_articles = int(outlet_article_totals.loc[outlet_label])
        n_topic_articles = int(outlet_topic_totals.loc[outlet_label])
        corpus_share = n_articles / corpus_article_total if corpus_article_total else float("nan")

        if n_topic_articles > 0:
            probs = counts / n_topic_articles
            nonzero_probs = probs[probs > 0]
            # Entropy shape is size-independent, but estimates are noisier for n < 1000.
            entropy = float(-(nonzero_probs * nonzero_probs.map(math.log)).sum())
            entropy_normalized = entropy / math.log(total_topics) if total_topics > 1 else 0.0
        else:
            probs = counts * 0.0
            entropy = float("nan")
            entropy_normalized = float("nan")

        covered_topics = int((counts >= min_articles).sum())
        coverage_breadth_raw = covered_topics / total_topics if total_topics else float("nan")

        expected_covered = 0.0
        if n_topic_articles > 0 and total_topics > 0:
            for p_topic in corpus_topic_share.tolist():
                expected_covered += _binomial_tail_prob_ge_k(
                    n_topic_articles,
                    float(p_topic),
                    min_articles,
                    scipy_state,
                )
        coverage_breadth_expected = expected_covered / total_topics if total_topics else float("nan")
        # Primary H1 breadth measure — size-controlled.
        coverage_breadth_relative = (
            coverage_breadth_raw / coverage_breadth_expected
            if coverage_breadth_expected and not pd.isna(coverage_breadth_expected)
            else float("nan")
        )

        if n_topic_articles > 0:
            smooth_counts = counts + 1.0
            smooth_probs = smooth_counts / smooth_counts.sum()
            ratio = smooth_probs / tag_probs
            # KL compares distribution shapes, not sizes — relatively robust, but noisier for small n.
            kl_from_tagesschau = float((smooth_probs * ratio.map(math.log)).sum())
            if outlet_label == tagesschau_label:
                kl_from_tagesschau = 0.0
        else:
            kl_from_tagesschau = float("nan")

        qualifying_counts = counts[counts >= min_articles]
        if qualifying_counts.empty:
            topic_dominance_score = float("nan")
        else:
            # Size-controlled dominance — best for TYPE B detection.
            topic_dominance_score = float(
                (qualifying_counts / topic_totals.reindex(qualifying_counts.index)).mean()
            )

        rows.append(
            {
                "outlet_label": outlet_label,
                "n_articles": n_articles,
                "corpus_share": corpus_share,
                "entropy": entropy,
                "entropy_normalized": entropy_normalized,
                "coverage_breadth_raw": coverage_breadth_raw,
                "coverage_breadth_expected": coverage_breadth_expected,
                "coverage_breadth_relative": coverage_breadth_relative,
                "kl_from_tagesschau": kl_from_tagesschau,
                "topic_dominance_score": topic_dominance_score,
            }
        )

    return pd.DataFrame(rows).sort_values("outlet_label").reset_index(drop=True)
    # H1 evidence type: BOTH


def compute_topic_overlap_with_tagesschau(merged_articles, top_n=10) -> pd.DataFrame:
    """Computes top-topic overlap with Tagesschau for each alternative outlet.

    Low overlap indicates agenda divergence from the mainstream reference.
    The comparison is based on ranked topic lists rather than raw counts,
    so it is not confounded by outlet corpus size in the same way as raw
    coverage counts.

    Thesis relevance:
        Simplest direct test of H1 — do alternative outlets prioritize the
        same topics as the mainstream reference?

    Args:
        merged_articles: Article-level merged BERTopic assignments.
        top_n: Number of top topics per outlet to compare.

    Returns:
        pd.DataFrame: One row per non-Tagesschau outlet with overlap metrics.
    """
    topic_df = merged_articles.loc[
        merged_articles["merged_topic"] != -1,
        ["outlet_label", "merged_topic", "merged_display_label"],
    ].copy()
    columns = [
        "outlet_label",
        "top_n_topics",
        "tagesschau_top_n",
        "overlap_count",
        "overlap_share",
        "divergent_topics",
    ]
    if topic_df.empty:
        return pd.DataFrame(columns=columns)
        # H1 evidence type: BOTH

    label_map = {}
    for topic_id, label in (
        topic_df[["merged_topic", "merged_display_label"]]
        .drop_duplicates(subset=["merged_topic"])
        .itertuples(index=False)
    ):
        label_map[topic_id] = label if pd.notna(label) else str(topic_id)

    counts_df = (
        topic_df.groupby(["outlet_label", "merged_topic"])
        .size()
        .rename("n")
        .reset_index()
    )
    tagesschau_label = "Tagesschau"
    if tagesschau_label not in counts_df["outlet_label"].unique():
        raise KeyError("Tagesschau not found in merged_articles['outlet_label'].")

    tagesschau_top_ids = (
        counts_df.loc[counts_df["outlet_label"] == tagesschau_label]
        .sort_values(["n", "merged_topic"], ascending=[False, True])
        .head(top_n)["merged_topic"]
        .tolist()
    )
    tagesschau_top_labels = [label_map.get(topic_id, str(topic_id)) for topic_id in tagesschau_top_ids]

    rows = []
    for outlet_label in sorted(counts_df["outlet_label"].unique().tolist()):
        if outlet_label == tagesschau_label:
            continue

        outlet_top_ids = (
            counts_df.loc[counts_df["outlet_label"] == outlet_label]
            .sort_values(["n", "merged_topic"], ascending=[False, True])
            .head(top_n)["merged_topic"]
            .tolist()
        )
        overlap_ids = [topic_id for topic_id in outlet_top_ids if topic_id in tagesschau_top_ids]
        divergent_ids = [topic_id for topic_id in outlet_top_ids if topic_id not in tagesschau_top_ids]

        rows.append(
            {
                "outlet_label": outlet_label,
                "top_n_topics": [label_map.get(topic_id, str(topic_id)) for topic_id in outlet_top_ids],
                "tagesschau_top_n": tagesschau_top_labels,
                "overlap_count": len(overlap_ids),
                "overlap_share": len(overlap_ids) / len(outlet_top_ids) if outlet_top_ids else float("nan"),
                "divergent_topics": [label_map.get(topic_id, str(topic_id)) for topic_id in divergent_ids],
            }
        )

    return pd.DataFrame(rows).sort_values(["overlap_share", "outlet_label"]).reset_index(drop=True)
    # H1 evidence type: BOTH


def plot_outlet_topic_heatmap(
    merged_articles,
    top_n_topics=20,
    normalize="outlet",
    source_colors=None,
    measures_df=None,
    min_label=10,
):
    """Plots an outlet-by-topic heatmap for H1 analysis.

    Rows are outlets and columns are the most frequent merged topics in the
    corpus plus an "Other topics" column. The plot can emphasize within-outlet
    concentration or within-topic outlet dominance depending on the selected
    normalization.

    Thesis relevance:
        Single figure that shows both breadth (how many topics an outlet shows
        up in) and concentration (how strongly it clusters in a subset of
        topics) at the same time.

    Args:
        merged_articles: Article-level merged BERTopic assignments.
        top_n_topics: Number of globally frequent merged topics to display.
        normalize: Either "outlet" or "topic".
        source_colors: Optional mapping from outlet label to text color.
        measures_df: Optional output of compute_outlet_topic_measures used to
            sort outlets by entropy_normalized ascending.
        min_label: Minimum raw count required before a cell is annotated.

    Returns:
        matplotlib.figure.Figure: The rendered heatmap figure.
    """
    import numpy as np

    if normalize not in {"outlet", "topic"}:
        raise ValueError("normalize must be either 'outlet' or 'topic'.")

    topic_df = merged_articles.loc[
        merged_articles["merged_topic"] != -1,
        ["outlet_label", "merged_topic", "merged_display_label"],
    ].copy()

    fig, ax = plt.subplots(figsize=(18, 7), dpi=150)
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    if topic_df.empty:
        ax.text(0.5, 0.5, "No non-outlier merged topics available.", ha="center", va="center", fontsize=12)
        ax.set_axis_off()
        fig.tight_layout()
        return fig
        # H1 evidence type: BOTH

    topic_label_map = {}
    for topic_id, label in (
        topic_df[["merged_topic", "merged_display_label"]]
        .drop_duplicates(subset=["merged_topic"])
        .itertuples(index=False)
    ):
        topic_label_map[topic_id] = label if pd.notna(label) else str(topic_id)

    all_outlets = sorted(topic_df["outlet_label"].dropna().unique().tolist())
    if measures_df is not None and {"outlet_label", "entropy_normalized"}.issubset(measures_df.columns):
        ranked = (
            measures_df[["outlet_label", "entropy_normalized"]]
            .drop_duplicates(subset=["outlet_label"])
            .sort_values(["entropy_normalized", "outlet_label"], ascending=[True, True])
        )
        outlet_order = [
            outlet_label
            for outlet_label in ranked["outlet_label"].tolist()
            if outlet_label in all_outlets
        ]
        outlet_order.extend(
            [outlet_label for outlet_label in all_outlets if outlet_label not in outlet_order]
        )
    else:
        outlet_order = all_outlets

    topic_totals = topic_df.groupby("merged_topic").size().sort_values(ascending=False)
    selected_topic_ids = topic_totals.head(top_n_topics).index.tolist()

    outlet_topic_counts = (
        topic_df.groupby(["outlet_label", "merged_topic"])
        .size()
        .unstack(fill_value=0)
        .reindex(index=outlet_order, fill_value=0)
        .astype(float)
    )
    outlet_non_outlier_totals = outlet_topic_counts.sum(axis=1)

    top_topic_counts = outlet_topic_counts.reindex(columns=selected_topic_ids, fill_value=0).copy()
    other_topic_counts = outlet_non_outlier_totals - top_topic_counts.sum(axis=1)

    heat_counts = top_topic_counts.copy()
    heat_counts["Other topics"] = other_topic_counts

    if normalize == "outlet":
        denom = outlet_non_outlier_totals.replace(0, pd.NA)
        heat_values = heat_counts.div(denom, axis=0).fillna(0.0)
        colorbar_label = "Share of outlet articles"
    else:
        topic_denoms = topic_totals.reindex(selected_topic_ids).replace(0, pd.NA)
        heat_values = top_topic_counts.div(topic_denoms, axis=1).fillna(0.0)
        heat_values["Other topics"] = other_topic_counts.div(
            outlet_non_outlier_totals.replace(0, pd.NA)
        ).fillna(0.0)
        colorbar_label = "Share of topic articles"

    column_labels = [topic_label_map.get(topic_id, str(topic_id)) for topic_id in selected_topic_ids] + ["Other topics"]
    heat_values.columns = column_labels
    heat_counts.columns = column_labels

    matrix = heat_values.to_numpy(dtype=float)
    vmax = float(matrix.max()) if matrix.size else 1.0
    if vmax <= 0:
        vmax = 1.0

    image = ax.imshow(matrix, aspect="auto", cmap="YlOrRd", vmin=0.0, vmax=vmax)
    ax.set_xticks(range(len(column_labels)))
    ax.set_xticklabels(column_labels, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(len(outlet_order)))
    ax.set_yticklabels(outlet_order, fontsize=10)

    if source_colors is not None:
        for tick in ax.get_yticklabels():
            if tick.get_text() in source_colors:
                tick.set_color(source_colors[tick.get_text()])

    ax.set_xticks(np.arange(-0.5, len(column_labels), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(outlet_order), 1), minor=True)
    ax.grid(which="minor", color="#FFFFFF", linewidth=1.0)
    ax.tick_params(which="minor", bottom=False, left=False)

    threshold = vmax * 0.55
    for row_idx in range(len(outlet_order)):
        for col_idx in range(len(column_labels)):
            raw_count = int(heat_counts.iat[row_idx, col_idx])
            if raw_count < min_label:
                continue
            value = float(heat_values.iat[row_idx, col_idx])
            text_color = "white" if value >= threshold and value > 0 else "black"
            ax.text(
                col_idx,
                row_idx,
                f"{raw_count}",
                ha="center",
                va="center",
                fontsize=8,
                color=text_color,
            )

    ax.set_title(
        f"Outlet × Topic Heatmap in Merged Topic Space ({normalize}-normalized)",
        fontsize=14,
        pad=14,
    )
    colorbar = fig.colorbar(image, ax=ax, fraction=0.025, pad=0.02)
    colorbar.set_label(colorbar_label)
    fig.tight_layout()
    return fig
    # H1 evidence type: BOTH


def print_imbalance_report(measures_df) -> None:
    """Prints a formatted corpus-imbalance diagnostic table.

    Run this before reporting H1 results so that outlets with noisier
    estimates are flagged explicitly in the methods section.

    Args:
        measures_df: Output of compute_outlet_topic_measures.

    Returns:
        None
    """
    required_columns = [
        "outlet_label",
        "n_articles",
        "corpus_share",
        "entropy_normalized",
        "coverage_breadth_relative",
        "kl_from_tagesschau",
        "topic_dominance_score",
    ]
    missing = [column for column in required_columns if column not in measures_df.columns]
    if missing:
        raise KeyError(f"measures_df is missing required columns: {missing}")

    def _fmt_float(value):
        return "nan" if pd.isna(value) else f"{value:.3f}"

    working = measures_df.loc[:, required_columns].copy()
    working = working.sort_values(["n_articles", "outlet_label"], ascending=[False, True]).reset_index(drop=True)

    header = (
        f"{'outlet':<20} {'n_articles':>10} {'corpus_share':>12} {'flag':>9} "
        f"{'entropy_norm':>14} {'breadth_rel':>14} {'kl_from_ts':>12} {'dominance':>12}"
    )
    print(header)
    print("-" * len(header))

    for row in working.itertuples(index=False):
        flag = "⚠ SMALL" if row.n_articles < 1000 else "✓ OK"
        print(
            f"{str(row.outlet_label):<20} "
            f"{int(row.n_articles):>10,} "
            f"{row.corpus_share:>12.1%} "
            f"{flag:>9} "
            f"{_fmt_float(row.entropy_normalized):>14} "
            f"{_fmt_float(row.coverage_breadth_relative):>14} "
            f"{_fmt_float(row.kl_from_tagesschau):>12} "
            f"{_fmt_float(row.topic_dominance_score):>12}"
        )

    print()
    print(
        "Measures most robust to corpus imbalance: coverage_breadth_relative, "
        "topic_dominance_score (both size-controlled). Use entropy and KL "
        "divergence for large outlets only, or report with caveat for n < 1000."
    )
    # H1 evidence type: BOTH


In [ ]:
# Run the H1 analysis and show the actual outputs
moa = importlib.reload(moa)

H1_SOURCE_COLORS = {
    "Antispiegel": "#0072B2",
    "Compact": "#E69F00",
    "Deutschlandkurier": "#009E73",
    "Nius": "#D55E00",
    "RT": "#CC79A7",
    "Tagesschau": "#56B4E9",
    "Tichys Einblick": "#F0E442",
}

H1_RESULTS = moa.run_h1_analysis(
    merged_articles,
    source_colors=H1_SOURCE_COLORS,
    top_n_topics=20,
    top_n_overlap=10,
    min_articles=10,
    save_dir=None,
)

display(
    H1_RESULTS["measures_df"]
    .sort_values(["coverage_breadth_relative", "entropy_normalized", "outlet_label"], ascending=[True, True, True])
    .reset_index(drop=True)
)

display(
    H1_RESULTS["overlap_df"]
    .sort_values(["overlap_share", "outlet_label"], ascending=[True, True])
    .reset_index(drop=True)
)


## 3D Semantic Space Visualization

The 2D UMAP projections above necessarily compress the high-dimensional embedding space, which can introduce visual artifacts such as artificial cluster overlap or separation. A 3D projection retains one additional degree of freedom, preserving more of the local neighborhood structure from the original embedding manifold.

The interactive 3D plot below serves two purposes:
1. **Robustness check**: confirms that the spatial patterns visible in the 2D footprint maps are not projection artifacts — clusters that appear separate in 2D should remain separate in 3D, and vice versa.
2. **Exploratory depth**: allows rotating the semantic space to inspect outlet distributions from angles that the fixed 2D view cannot show — particularly useful for outlets whose footprints overlap heavily in the 2D plane but may separate along the third axis.

In [ ]:
# 3D UMAP projection of the merged semantic space
import plotly.express as px

# Extract embeddings and fit a 3D UMAP reducer
docs_3d = combined_prepared["document"].tolist()
embeddings_3d = merged_model._extract_embeddings(docs_3d, method="document")

reducer_3d = UMAP(
    n_neighbors=10,
    n_components=3,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)
coords_3d = reducer_3d.fit_transform(embeddings_3d)

merged_articles["umap_3d_x"] = coords_3d[:, 0]
merged_articles["umap_3d_y"] = coords_3d[:, 1]
merged_articles["umap_3d_z"] = coords_3d[:, 2]

print(f"3D UMAP fitted: {coords_3d.shape[0]} documents → 3 components")

In [ ]:
# 3D scatter: all outlets colored by source
OUTLET_COLOR_MAP = {
    "Tagesschau": "#4878CF",
    "RT": "#B22222",
    "Antispiegel": "#D4472A",
    "Tichys Einblick": "#6A994E",
    "Nius": "#E07B39",
    "Compact": "#7B5EA7",
    "Deutschlandkurier": "#3A7D7B",
}

plot_3d = merged_articles[["outlet_label", "merged_display_label", "umap_3d_x", "umap_3d_y", "umap_3d_z"]].copy()
plot_3d["outlet_label"] = pd.Categorical(
    plot_3d["outlet_label"],
    categories=["Tagesschau", "RT", "Antispiegel", "Tichys Einblick", "Nius", "Compact", "Deutschlandkurier"],
    ordered=True,
)
plot_3d = plot_3d.sort_values("outlet_label").reset_index(drop=True)

fig_3d_outlet = px.scatter_3d(
    plot_3d,
    x="umap_3d_x",
    y="umap_3d_y",
    z="umap_3d_z",
    color="outlet_label",
    color_discrete_map=OUTLET_COLOR_MAP,
    hover_data=["merged_display_label"],
    opacity=0.35,
    title="3D Merged Semantic Space — Articles by Outlet",
    labels={
        "umap_3d_x": "UMAP-1",
        "umap_3d_y": "UMAP-2",
        "umap_3d_z": "UMAP-3",
        "outlet_label": "Outlet",
    },
)
fig_3d_outlet.update_traces(marker_size=2.5)
fig_3d_outlet.update_layout(
    width=1000,
    height=750,
    legend_title_text="Outlet",
    scene=dict(
        xaxis_title="UMAP-1",
        yaxis_title="UMAP-2",
        zaxis_title="UMAP-3",
    ),
)
fig_3d_outlet.show()

In [ ]:
# 3D scatter: colored by merged topic (top 20)
top_20_topics = (
    merged_topic_info_display
    .loc[merged_topic_info_display["Topic"] != -1]
    .nsmallest(20, "DisplayTopic")["Topic"]
    .tolist()
)

plot_3d_topic = merged_articles[["merged_topic", "merged_display_label", "outlet_label", "umap_3d_x", "umap_3d_y", "umap_3d_z"]].copy()
plot_3d_topic["topic_label"] = plot_3d_topic["merged_display_label"].where(
    plot_3d_topic["merged_topic"].isin(top_20_topics),
    other="Other / Outlier",
)

fig_3d_topic = px.scatter_3d(
    plot_3d_topic,
    x="umap_3d_x",
    y="umap_3d_y",
    z="umap_3d_z",
    color="topic_label",
    hover_data=["outlet_label", "merged_display_label"],
    opacity=0.35,
    title="3D Merged Semantic Space — Top 20 Topics",
    labels={
        "umap_3d_x": "UMAP-1",
        "umap_3d_y": "UMAP-2",
        "umap_3d_z": "UMAP-3",
        "topic_label": "Topic",
    },
)
fig_3d_topic.update_traces(marker_size=2.5)
fig_3d_topic.update_layout(
    width=1000,
    height=750,
    legend_title_text="Merged Topic",
    scene=dict(
        xaxis_title="UMAP-1",
        yaxis_title="UMAP-2",
        zaxis_title="UMAP-3",
    ),
)
fig_3d_topic.show()

## Interpretation Guide

### Reading the 3D Semantic Space

- **Outlet-colored view**: Shows how each outlet distributes across the shared semantic manifold. Tagesschau (blue) should appear as a broadly dispersed cloud if it functions as the agenda-setting mainstream reference. Alternative outlets with narrower agendas will appear as tighter, more localized clusters — consistent with TYPE A (breadth restriction) from H1.
- **Topic-colored view**: Confirms that the merged topic clusters form spatially coherent regions in 3D, validating the merge quality. Topics that appeared merged or overlapping in the 2D projection may resolve into distinct clusters when the third dimension is available.

### What to Look For

| Pattern | H1 Interpretation |
|---------|-------------------|
| An outlet's cloud spans most of the 3D volume | Broad agenda, low distortion |
| An outlet clusters in 1–2 dense regions only | TYPE A — breadth restriction |
| An outlet spans broadly but with extreme density peaks | TYPE B — concentration distortion |
| Two outlets occupy the same 3D region | Shared agenda emphasis (ideological alignment?) |
| An outlet occupies a region with few Tagesschau points | Agenda divergence — topics not prioritized by mainstream |

### Adding Tagesschau to the Semantic Footprint Maps

The 2D footprint maps above now include Tagesschau as the first plot in the series. This serves as a **visual baseline**: its KDE cloud and coverage statistics provide the reference against which alternative outlets should be compared. If Tagesschau's footprint is broad and diffuse while an alternative outlet's footprint is narrow and peaked, that contrast directly visualizes the agenda distortion that the quantitative H1 measures detect.

## Academic Validity and Methodological Notes

### Dimensionality Reduction Caveats

UMAP projections (both 2D and 3D) are **topology-preserving approximations**, not exact distance-preserving embeddings. This means:
- **Local neighborhoods** are preserved with high fidelity — nearby points in the original embedding space remain nearby in the projection (McInnes et al., 2018).
- **Global distances** between distant clusters should be interpreted with caution — the absolute distance between two topic clusters in the plot does not map linearly to semantic distance.
- The 3D projection preserves strictly more structure than 2D (higher trustworthiness and continuity scores), but it remains an approximation.

### Why Both 2D and 3D

The 2D projections are better for **publication figures** — they are static, reproducible, and can be included in the thesis PDF. The 3D projections are better for **exploratory analysis** — they allow the researcher to rotate the space and discover patterns that the 2D projection may obscure. Using both is standard practice in embedding visualization research (e.g., BERTopic documentation recommends 2D for display, while 3D is used for validation).

### Validity of the Merged Model Approach

The thesis uses **post hoc model merging** rather than fitting a single BERTopic model on the pooled corpus. This is methodologically important because:
1. **Outlet-specific topic structures are preserved** — each outlet's topics are first identified in isolation, then aligned across outlets.
2. **The merge threshold** (`min_similarity=0.7`) controls how aggressively topics from different outlets are collapsed into shared topics. This is a researcher degree of freedom that should be reported and sensitivity-tested.
3. The merged space should be described as a *post hoc aligned semantic space*, not a jointly estimated topic model.

### Semantic Footprint Map — Methodological Position

The semantic footprint map is a **custom comparative visualization** that combines established techniques:
- UMAP document projection (McInnes et al., 2018)
- KDE density overlay (Rosenblatt, 1956)
- One-vs-all highlighting against a corpus background

The combination is tailored to the comparative agenda analysis task of this thesis. The closest methodological precedents are BERTopic's own `visualize_documents()` and UMAP+KDE contour approaches used in embedding space analysis (e.g., Wang et al., 2023, DiffusionDB). The figure should be cited as a custom research visualization, not as a standard named plot type.

### References
- McInnes, L., Healy, J., Saul, N., & Großberger, L. (2018). UMAP: Uniform Manifold Approximation and Projection. *JOSS*, 3(29), 861.
- Grootendorst, M. (2022). BERTopic: Neural topic modeling with a class-based TF-IDF procedure. *arXiv:2203.05794*.
- Rosenblatt, M. (1956). Remarks on Some Nonparametric Estimates of a Density Function. *Annals of Mathematical Statistics*, 27(3), 832–837.
- Wang, Z. J., et al. (2023). DiffusionDB: A Large-Scale Prompt Gallery Dataset for Text-to-Image Generative Models. *ACL 2023*.